# 03 · Data audit

How small are the objects, how dark are the scenes, and how well are the two cameras aligned? The alignment result also sets the synthetic shifts of the probe in notebook 09.

| | |
|---|---|
| **Kaggle settings** | Accelerator **GPU T4 x2**, Internet **on** |
| **Inputs to attach** | `cffm-net-pilot`, raw LLVIP and M3FD (or the output of 02) |
| **Expected runtime** | about 5 minutes |
| **Produces** | `runs/audit/audit.json` and the figures below |

In [ ]:
# Same setup cell in every notebook: find the project, install it, check the GPU.
import glob, shutil, subprocess, sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    PROJECT = Path("/kaggle/working/cffm-net-pilot")
    if not PROJECT.exists():
        # Kaggle mounts datasets at different depths under /kaggle/input, so search five levels.
        hits = [Path(p).parent for k in range(1, 6) for p in glob.glob("/kaggle/input" + "/*" * k + "/pyproject.toml")
                if (Path(p).parent / "src" / "cffm").is_dir()]
        # Earlier notebooks' outputs hold stale copies of the code, so prefer the uploaded dataset.
        hits.sort(key=lambda h: any((h.parent / d).exists() for d in ("runs", "data")))
        zips = [Path(p) for k in range(1, 5) for p in glob.glob("/kaggle/input" + "/*" * k + ".zip")
                if "cffm" in Path(p).name]
        if hits:
            shutil.copytree(hits[0], PROJECT, copy_function=shutil.copyfile)
        elif zips:                       # the zip was attached but not unpacked by Kaggle
            shutil.unpack_archive(str(zips[0]), "/kaggle/working")   # it holds one folder, cffm-net-pilot/
        assert PROJECT.exists(), "Add the 'cffm-net-pilot' dataset (the uploaded zip) as an input to this notebook."
        for p in [PROJECT, *PROJECT.rglob("*")]:     # inputs are read-only; our copy must be writable
            p.chmod(p.stat().st_mode | 0o200)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.171",
                    "faster-coco-eval>=1.6.7", "cloudpickle", "pytest"], check=True)   # cloudpickle: two-GPU launcher
    # Editable install, so DataLoader and DDP worker processes can import cffm too.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(PROJECT)], check=True)
else:
    PROJECT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "cffm").is_dir())
sys.path.insert(0, str(PROJECT / "src"))

import cffm
from cffm import env as cenv, pipeline
print("cffm", cffm.__version__, "imported from", Path(cffm.__file__).parent)   # must be inside PROJECT
env = cenv.setup()                       # also moves into the project folder
plan = pipeline.load_plan(env.project)   # configs/pilot.yaml

In [ ]:
for name in ("llvip", "m3fd"):
    pipeline.prepare(name, env, plan)

## 1. Object sizes at training resolution

Size is the square root of box area at 640 px; dashed lines mark the AI-TOD bands (8, 16, 32 px) and COCO's 96 px boundary.

In [ ]:
import cv2, json, numpy as np, matplotlib.pyplot as plt
audit = {}
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
for ax, name in zip(axes, ("llvip", "m3fd")):
    root = env.data / name
    card = json.loads((root / "dataset_card.json").read_text())
    H, W = cv2.imread(str(next((root / "images" / "visible" / "train").glob("*.jpg")))).shape[:2]   # stored size
    sizes, per_image, classes = [], [], np.zeros(len(card["names"]), int)
    for f in (root / "labels" / "visible" / "train").glob("*.txt"):
        lines = [l.split() for l in f.read_text().splitlines() if l.strip()]
        per_image.append(len(lines))
        for c, x, y, w, h in lines:
            classes[int(c)] += 1
            sizes.append(np.sqrt(float(w) * W * float(h) * H))       # labels are normalised to the image
    sizes = np.array(sizes)
    ax.hist(sizes, bins=np.logspace(0, 2.6, 60), color="#2F6DB5", alpha=0.8)
    ax.set_xscale("log"); ax.set_title(f"{name}: {len(sizes)} training objects"); ax.set_xlabel("object size (px)")
    for t in (8, 16, 32, 96): ax.axvline(t, ls="--", c="gray", lw=0.8)
    audit[name] = {"median_size_px": float(np.median(sizes)), "frac_below_16px": float((sizes < 16).mean()),
                   "frac_below_32px": float((sizes < 32).mean()), "objects_per_image": float(np.mean(per_image)),
                   "class_counts": dict(zip(card["names"], classes.tolist()))}
plt.tight_layout(); plt.show()
print(json.dumps(audit, indent=2))

## 2. How dark are the scenes?

Mean visible brightness; LLVIP is mostly night, where thermal should earn its place.

In [ ]:
import cv2
fig, axes = plt.subplots(1, 2, figsize=(14, 3.2))
for ax, name in zip(axes, ("llvip", "m3fd")):
    files = sorted((env.data / name / "images" / "visible" / "val").glob("*.jpg"))[::5]
    b = np.array([cv2.imread(str(f), cv2.IMREAD_GRAYSCALE).mean() for f in files])
    ax.hist(b, bins=40, color="#C0622B", alpha=0.8); ax.set_title(f"{name}: mean visible brightness (0-255)")
    audit[name]["frac_dark_below_50"] = float((b < 50).mean())
plt.tight_layout(); plt.show()

## 3. How well are the cameras aligned?

Phase correlation of Sobel edge maps gives the shift that best lines up each pair, (0, 0) if registered.
Edges are used because outlines appear in both sensors even where intensities differ.

In [ ]:
def edges(img):
    img = cv2.GaussianBlur(img.astype(np.float32), (5, 5), 0)
    return cv2.magnitude(cv2.Sobel(img, cv2.CV_32F, 1, 0), cv2.Sobel(img, cv2.CV_32F, 0, 1))

rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
for ax, name in zip(axes, ("llvip", "m3fd")):
    files = sorted((env.data / name / "images" / "visible" / "val").glob("*.jpg"))
    shifts = []
    for f in rng.choice(files, size=min(200, len(files)), replace=False):
        v = edges(cv2.imread(str(f), cv2.IMREAD_GRAYSCALE))
        t = edges(cv2.imread(str(env.data / name / "images" / "infrared" / "val" / f.name), cv2.IMREAD_GRAYSCALE))
        win = cv2.createHanningWindow(v.shape[::-1], cv2.CV_32F)
        (dx, dy), resp = cv2.phaseCorrelate(v, t, win)
        if resp > 0.05:                       # too little shared structure to measure
            shifts.append(np.hypot(dx, dy))
    shifts = np.array(shifts)
    ax.hist(shifts, bins=40, color="#3B8B66", alpha=0.85)
    ax.set_title(f"{name}: residual misalignment, {len(shifts)} pairs"); ax.set_xlabel("shift (px at 640)")
    audit[name]["misalignment_px"] = {q: float(np.percentile(shifts, q)) for q in (50, 90, 99)}
plt.tight_layout(); plt.show()
print({n: a["misalignment_px"] for n, a in audit.items()})

**Reading it.** If the 90th percentile is a couple of pixels, the benchmark is registered, and the 4, 8 and 16 px shifts of notebook 09 test robustness well beyond the data.

In [ ]:
(env.runs / "audit").mkdir(parents=True, exist_ok=True)
(env.runs / "audit" / "audit.json").write_text(json.dumps(audit, indent=2))
print(json.dumps(audit, indent=2))